# EndoScan — AR Stop-2 coverage diagnostic (CoMPARA × LINCS)

**Operator: just `Runtime → Run all`, then paste back the single `COPY-BACK SUMMARY`
block printed by the last cell.** No editing, no tokens, no Google Drive, no 20 GB
download. Runs in a few minutes.

This notebook answers one question for the M6 androgen-receptor (AR) endpoint, the same
way we did for ER: **how many AR-labelled compounds have a LINCS signature, and what is
the positive/negative split — versus the quality gate's floors `min_overlap=40` and
`min_compounds_per_class=20`** — across candidate LINCS cell-line contexts.

It uses ONLY:
- CoMPARA's **EXPERIMENTAL / MEASURED** AR activity set (EPA figshare article 10321697,
  Mansouri et al. 2020) — **never** the consensus-QSAR *predictions* for the ~55k library
  (article 10322012 is explicitly avoided; predicted columns are filtered out).
- LINCS L1000 **metadata only** (`sig_info` + `pert_info`, GSE92742) — no expression
  matrix (`gctx`) is downloaded; overlap is a metadata join, like the ER Stop-2 guard.

Nothing is trained, staged, registered, or written back to the repo. If the data fetch
fails, the last cell prints a clear per-source diagnostic instead of a cryptic error.


In [ ]:
# 1) VERSION STAMP (first printed line — confirms which notebook ran), then install deps.
BUILD_TAG = "EndoScan AR Stop-2 diagnostic | build R2 (robust-fetch + stamp + reachability) | 2026-06-26"
print("================================================================")
print("NOTEBOOK VERSION:", BUILD_TAG)
print("================================================================")

import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "pandas", "requests", "pyarrow", "openpyxl", "rdkit"], check=True)

import io, gzip, re, json, time, zipfile
from pathlib import Path
from collections import defaultdict
import requests
import pandas as pd
from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import inchi as rd_inchi
RDLogger.DisableLog("rdApp.*")
print("imports OK | pandas", pd.__version__)


In [ ]:
# 2) CONFIG — no editing required. (FORCE_* overrides default to auto-detect.)
# Gate floors mirror registry/data/quality_gates.yaml (the agent reads them read-only).
MIN_OVERLAP = 40
MIN_COMPOUNDS_PER_CLASS = 20

# CoMPARA EXPERIMENTAL set (EPA figshare). 10321697 = "Training Set+Prediction Set"
# (the curated, MEASURED AR activity calls used to build the models); 10321994 =
# "Data Associated with Publication" (fallback). 10322012 = "Consensus models"
# (PREDICTED outputs) — explicitly EXCLUDED.
COMPARA_EXPERIMENTAL_ARTICLES = [10321697, 10321994]
COMPARA_CONSENSUS_PREDICTION_ARTICLE = 10322012  # never used

# Best-effort EPA gAFTP fallback roots (exact CoMPARA subdir not guaranteed; probed + logged).
COMPARA_GAFTP_CANDIDATES = [
    "https://gaftp.epa.gov/COMPTOX/Sustainable_Chemistry_Data/CoMPARA_QSAR_Models/",
    "https://gaftp.epa.gov/COMPTOX/Sustainable_Chemistry_Data/",
]

# LINCS L1000 metadata (GSE92742) — small .txt.gz, fetched directly (no gctx).
LINCS_SIG_INFO = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_sig_info.txt.gz"
LINCS_PERT_INFO = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_pert_info.txt.gz"

# Candidate cell-line contexts (UPPERCASE LINCS cell_id). VCaP/LNCaP are androgen-
# responsive; MCF7/A549 are what ER used; the broad set adds PC3.
CELL_LINES_PROBE = ["VCAP", "LNCAP", "MCF7", "A549", "PC3"]
CONTEXTS = {
    "VCaP (androgen)":             ["VCAP"],
    "VCaP+LNCaP (androgen)":       ["VCAP", "LNCAP"],
    "MCF7/A549 (ER-style)":        ["MCF7", "A549"],
    "broad (VCaP+MCF7+A549+PC3)":  ["VCAP", "MCF7", "A549", "PC3"],
}

# HTTP robustness + optional pins (left None for hands-off Run-all).
HTTP_HEADERS = {"User-Agent": "Mozilla/5.0 (EndoScan AR Stop-2 diagnostic; research use)"}
HTTP_RETRIES = 3
FORCE_FILE = None        # e.g. "AR_training.xlsx" (substring of the table key)
FORCE_CALL_COL = None    # e.g. "AR_Binding"
FORCE_STRUCT_COL = None  # e.g. "InChI"

# Column-name heuristics.
RE_PRED = re.compile(r"pred|consensus|qsar|model|score|prob|dock|_p_|applicab", re.I)
RE_CALL = re.compile(r"bind|agonist|antagonist|activ|call|class|outcome", re.I)
RE_STRUCT_INCHI = re.compile(r"inchi(?!key)", re.I)
RE_STRUCT_SMILES = re.compile(r"smiles|canonical", re.I)
print("config OK | floors: overlap>=%d, per-class>=%d" % (MIN_OVERLAP, MIN_COMPOUNDS_PER_CLASS))


In [ ]:
# 3) SOURCE — robustly fetch CoMPARA EXPERIMENTAL files (figshare API -> ndownloader ->
#    gAFTP -> fallback article), handle .zip/.xlsx/.csv/.tsv/.sdf, LOG every attempt, and
#    auto-select the MEASURED AR activity table+column (predicted/consensus excluded).
WORK = Path("/content/compara"); WORK.mkdir(parents=True, exist_ok=True)
ATTEMPT_LOG = []  # (source, ok, detail)
SESS = requests.Session(); SESS.headers.update(HTTP_HEADERS)


def _log(src, ok, detail):
    ATTEMPT_LOG.append((str(src), bool(ok), str(detail)))


def http_get(url, timeout=600):
    last = None
    for i in range(HTTP_RETRIES):
        try:
            return SESS.get(url, timeout=timeout, allow_redirects=True)
        except Exception as exc:  # transient network — retry with backoff
            last = exc; time.sleep(2 ** i)
    raise last


def fetch_to(url, dest):
    if not url:
        return None
    try:
        r = http_get(url)
    except Exception as exc:
        _log(url, False, "request error %s" % exc); return None
    if r.status_code != 200:
        _log(url, False, "HTTP %s" % r.status_code); return None
    dest.write_bytes(r.content)
    _log(url, True, "%d bytes -> %s" % (len(r.content), dest.name))
    return dest


def figshare_reachability():
    """FIRST network action — print the figshare API HTTP status so the operator sees
    immediately whether figshare is reachable from Colab (a ~1s empty result => the
    fetch loop never made a network call / figshare unreachable)."""
    url = "https://api.figshare.com/v2/articles/%d" % COMPARA_EXPERIMENTAL_ARTICLES[0]
    try:
        r = http_get(url, timeout=60)
        n = len(r.content)
        print("FIGSHARE REACHABILITY: %s -> HTTP %s (%d bytes)" % (url, r.status_code, n))
        _log(url, r.status_code == 200, "reachability HTTP %s (%d bytes)" % (r.status_code, n))
    except Exception as exc:
        print("FIGSHARE REACHABILITY: %s -> ERROR %s" % (url, exc))
        _log(url, False, "reachability error %s" % exc)


figshare_reachability()


def figshare_list(article_id):
    url = "https://api.figshare.com/v2/articles/%s" % article_id
    try:
        r = http_get(url, timeout=120)
    except Exception as exc:
        _log(url, False, "API error %s" % exc); return []
    if r.status_code != 200:
        _log(url, False, "API HTTP %s" % r.status_code); return []
    files = r.json().get("files", [])
    _log(url, True, "API listed %d file(s): %s" % (len(files), [f.get("name") for f in files]))
    return files


def is_zip(path):
    low = str(path).lower()
    if low.endswith((".xlsx", ".xls")):  # office files are PK zips too — treat as leaves
        return False
    try:
        return zipfile.is_zipfile(path)
    except Exception:
        return False


def expand(path, depth=0):
    """Leaf tabular paths; extract .zip recursively (xlsx/xls/csv/tsv/sdf are leaves)."""
    if depth > 3:
        return []
    if is_zip(path):
        out = []
        try:
            with zipfile.ZipFile(path) as z:
                names = [n for n in z.namelist() if not n.endswith("/")]
                _log(path.name, True, "zip contains: %s" % names[:20])
                exdir = path.parent / (path.name + "_x"); exdir.mkdir(exist_ok=True)
                for nm in names:
                    try:
                        out += expand(Path(z.extract(nm, exdir)), depth + 1)
                    except Exception as exc:
                        _log(nm, False, "extract failed %s" % exc)
        except Exception as exc:
            _log(path.name, False, "bad zip %s" % exc)
        return out
    if str(path).lower().endswith((".csv", ".tsv", ".txt", ".tab", ".xlsx", ".xls", ".sdf")):
        return [path]
    return []


def load_frames(path):
    name = path.name; low = name.lower()
    try:
        if low.endswith(".csv"):
            return {name: pd.read_csv(path, low_memory=False)}
        if low.endswith((".tsv", ".txt", ".tab")):
            return {name: pd.read_csv(path, sep=None, engine="python", low_memory=False)}
        if low.endswith((".xlsx", ".xls")):
            return {"%s::%s" % (name, s): df for s, df in pd.read_excel(path, sheet_name=None).items()}
        if low.endswith(".sdf"):
            rows = []
            for mol in Chem.SDMolSupplier(str(path)):
                if mol is None:
                    continue
                d = dict(mol.GetPropsAsDict())
                try:
                    d["__InChI__"] = rd_inchi.MolToInchi(mol)
                except Exception:
                    d["__InChI__"] = None
                rows.append(d)
            return {name: pd.DataFrame(rows)} if rows else {}
    except Exception as exc:
        _log(name, False, "parse error %s" % exc)
    return {}


def ingest(path, source_tag, frames):
    for leaf in expand(path):
        for key, df in load_frames(leaf).items():
            frames[(source_tag, key)] = df
            _log(key, True, "loaded rows=%d cols=%d" % (len(df), df.shape[1]))


frames = {}

# (0) operator-uploaded files in /content/compara (last-resort manual path; no editing needed)
for p in sorted(WORK.glob("*")):
    if p.is_file() and not RE_PRED.search(p.name):
        ingest(p, "uploaded", frames)

# (1) figshare API -> download_url, falling back to ndownloader/<id>
if not frames:
    for art in COMPARA_EXPERIMENTAL_ARTICLES:
        for f in figshare_list(art):
            fn = f.get("name", "file")
            if RE_PRED.search(fn):
                _log(fn, False, "skip predicted/consensus by name"); continue
            dest = WORK / fn
            ok = fetch_to(f.get("download_url"), dest)
            if ok is None and f.get("id"):
                ok = fetch_to("https://ndownloader.figshare.com/files/%s" % f["id"], dest)
            if ok is not None:
                ingest(dest, "figshare:%s" % art, frames)
        if frames:
            break

# (2) EPA gAFTP mirror (best-effort directory scrape)
if not frames:
    for base in COMPARA_GAFTP_CANDIDATES:
        try:
            idx = http_get(base, timeout=120)
        except Exception as exc:
            _log(base, False, "gaftp error %s" % exc); continue
        _log(base, idx.status_code == 200, "gaftp index HTTP %s" % idx.status_code)
        if idx.status_code != 200:
            continue
        hrefs = re.findall(r'href="([^"]+\.(?:zip|csv|xlsx|xls|tsv|txt))"', idx.text, re.I)
        for h in hrefs:
            if RE_PRED.search(h):
                continue
            u = h if h.startswith("http") else base.rstrip("/") + "/" + h.lstrip("/")
            dest = WORK / Path(h).name
            if fetch_to(u, dest) is not None:
                ingest(dest, "gaftp", frames)
        if frames:
            break

print("=== FETCH ATTEMPT LOG ===")
for src, ok, detail in ATTEMPT_LOG:
    print(("  OK   " if ok else "  FAIL ") + "| %-52s | %s" % (src[:52], detail))
print("FILES DISCOVERED (loaded tables):", [k[1] for k in frames])

if not frames:
    bar = "=" * 72
    fails = ["%s: %s" % (s[:46], d) for s, ok, d in ATTEMPT_LOG if not ok]
    blob = " ".join(fails).lower()
    blocker = ("figshare API/download blocked (403/redirect) for all URLs"
               if ("403" in blob or "http 4" in blob or "api http" in blob)
               else "downloaded files had no recognized tabular content (see zip contents above)"
               if any("zip contains" in d for _, _, d in ATTEMPT_LOG)
               else "no source reachable / empty responses")
    print("\n" + bar)
    print("COPY-BACK SUMMARY  (AR Stop-2 — FETCH FAILED; nothing loaded)")
    print(bar)
    print("No CoMPARA EXPERIMENTAL table could be loaded.")
    print("Articles tried (experimental):", COMPARA_EXPERIMENTAL_ARTICLES,
          "| consensus EXCLUDED:", COMPARA_CONSENSUS_PREDICTION_ARTICLE)
    print("Per-source failures:")
    for x in fails:
        print("  -", x)
    if not fails:
        print("  (no failures logged — sources returned no tabular files)")
    print("LIKELY BLOCKER:", blocker)
    print("NEXT: upload the CoMPARA experimental table to /content/compara/ and re-run, "
          "or set FORCE_FILE/FORCE_CALL_COL/FORCE_STRUCT_COL from any columns printed above.")
    print(bar)
    raise SystemExit("AR Stop-2 halted: no experimental table loaded (see FETCH ATTEMPT LOG).")


def score_struct_col(cols):
    inchi = [c for c in cols if RE_STRUCT_INCHI.search(str(c))]
    smiles = [c for c in cols if RE_STRUCT_SMILES.search(str(c))]
    return (inchi[0] if inchi else None), (smiles[0] if smiles else None)


def candidate_calls(cols):
    cands = [c for c in cols if RE_CALL.search(str(c)) and not RE_PRED.search(str(c))]

    def rank(c):
        s = str(c).lower()
        return (0 if "bind" in s else 1 if ("agonist" in s or "antagonist" in s) else 2, len(s))

    return sorted(cands, key=rank)


def looks_binary(series):
    vals = set(series.astype(str).str.strip().str.lower().unique()) - {"nan", "", "na", "none", "-", "-666"}
    active = {"active", "1", "1.0", "true", "positive", "agonist", "antagonist", "binder", "yes"}
    inactive = {"inactive", "0", "0.0", "false", "negative", "non-binder", "no"}
    return bool(vals & active) and bool(vals & inactive)


chosen = None
print("\n-- scanning loaded tables for a MEASURED AR activity call --")
for (tag, key), df in frames.items():
    cols = list(df.columns)
    inchi_c, smiles_c = score_struct_col(cols)
    if FORCE_STRUCT_COL and FORCE_STRUCT_COL in cols:
        inchi_c = FORCE_STRUCT_COL
    if not (inchi_c or smiles_c):
        continue
    calls = [FORCE_CALL_COL] if (FORCE_CALL_COL and FORCE_CALL_COL in cols) else candidate_calls(cols)
    for c in calls:
        if c in df.columns and looks_binary(df[c]):
            if FORCE_FILE and FORCE_FILE not in key:
                continue
            dist = dict(df[c].astype(str).str.lower().value_counts().head(6))
            print("  candidate: %s | struct=%s smiles=%s | CALL=%s | values=%s"
                  % (key, inchi_c, smiles_c, c, dist))
            if chosen is None:
                chosen = (tag, key, df, inchi_c, smiles_c, c)

if chosen is None:
    print("\nNO measured AR activity call auto-detected. Columns per loaded table:")
    for (tag, key), df in frames.items():
        print("  %s -> %s" % (key, list(df.columns)))
    raise SystemExit("AR Stop-2 halted: tables loaded but no binary AR call column found "
                     "(set FORCE_CALL_COL/FORCE_STRUCT_COL from the columns above).")

SRC_TAG, SRC_KEY, SRC_DF, INCHI_COL, SMILES_COL, CALL_COL = chosen
print("\nSELECTED measured experimental set:")
print("  source  :", SRC_TAG, "(EXPERIMENTAL — NOT consensus-prediction article %s)"
      % COMPARA_CONSENSUS_PREDICTION_ARTICLE)
print("  table   :", SRC_KEY)
print("  struct  :", INCHI_COL or SMILES_COL, "(InChI preferred; SMILES fallback)")
print("  AR call :", CALL_COL)


In [ ]:
# 4) IDENTITY — derive a full InChIKey per structure (RDKit, deterministic), map the
#    measured call to 0/1, collapse to one label per structure, record conflicts.
def to_binary(v):
    s = str(v).strip().lower()
    if s in {"active", "1", "1.0", "true", "positive", "agonist", "antagonist", "binder", "yes"}:
        return 1
    if s in {"inactive", "0", "0.0", "false", "negative", "non-binder", "no"}:
        return 0
    return None

def inchikey(row):
    val = row.get(INCHI_COL) if INCHI_COL else None
    mol = None
    if isinstance(val, str) and val.strip().startswith("InChI="):
        mol = rd_inchi.MolFromInchi(val.strip())
    if mol is None and SMILES_COL:
        sm = row.get(SMILES_COL)
        if isinstance(sm, str) and sm.strip():
            mol = Chem.MolFromSmiles(sm.strip())
    if mol is None:
        return None
    try:
        return rd_inchi.MolToInchiKey(mol)
    except Exception:
        return None

pairs, n_rows, n_unmapped_call, n_no_struct = [], 0, 0, 0
for _, row in SRC_DF.iterrows():
    n_rows += 1
    lab = to_binary(row.get(CALL_COL))
    if lab is None:
        n_unmapped_call += 1; continue
    ik = inchikey(row)
    if ik is None:
        n_no_struct += 1; continue
    pairs.append((ik, lab))

by_key = defaultdict(set)
for ik, lab in pairs:
    by_key[ik].add(lab)

clean, conflicts = {}, 0
for ik, labs in by_key.items():
    if len(labs) > 1:
        conflicts += 1; continue
    clean[ik] = next(iter(labs))

AR_LABELS = clean  # InChIKey -> 0/1
n_pos = sum(v == 1 for v in clean.values())
n_neg = sum(v == 0 for v in clean.values())
print("rows in measured table        :", n_rows)
print("  dropped: unmapped call=%d, no derivable structure=%d" % (n_unmapped_call, n_no_struct))
print("clean AR structures (InChIKey):", len(clean), "| pos=%d neg=%d" % (n_pos, n_neg))
print("intra-structure conflicts excluded:", conflicts)
assert clean, "No clean AR labels derived — inspect the selected column/encoding."


In [ ]:
# 5) LINCS — fetch sig_info + pert_info (metadata only); per-line trt_cp signature counts.
def fetch_gz_tsv(url, dest):
    if not dest.exists():
        r = http_get(url, timeout=600); r.raise_for_status(); dest.write_bytes(r.content)
    return pd.read_csv(dest, sep="\t", low_memory=False)

sig = fetch_gz_tsv(LINCS_SIG_INFO, WORK / "sig_info.txt.gz")
pert = fetch_gz_tsv(LINCS_PERT_INFO, WORK / "pert_info.txt.gz")
print("sig_info cols :", list(sig.columns)[:8], "...")
print("pert_info cols:", list(pert.columns)[:8], "...")

SIG_TYPE = "pert_type" if "pert_type" in sig.columns else None
SIG_CELL = "cell_id" if "cell_id" in sig.columns else None
SIG_PERT = "pert_id" if "pert_id" in sig.columns else None
PERT_ID = "pert_id" if "pert_id" in pert.columns else None
PERT_INCHI = "inchi_key" if "inchi_key" in pert.columns else (
    "inchi_key_prefix" if "inchi_key_prefix" in pert.columns else None)
assert all([SIG_TYPE, SIG_CELL, SIG_PERT, PERT_ID, PERT_INCHI]), "Unexpected LINCS columns."

def norm(k):
    if k is None: return None
    s = str(k).strip().upper()
    return None if s in {"", "-666", "NAN", "NA", "RESTRICTED"} else s

pert_to_ik = {}
for p, k in zip(pert[PERT_ID].astype(str), pert[PERT_INCHI]):
    nk = norm(k)
    if nk:
        pert_to_ik[p] = nk

trt = sig[sig[SIG_TYPE].astype(str) == "trt_cp"].copy()
trt["_CELL"] = trt[SIG_CELL].astype(str).str.upper()
print("\ntrt_cp signatures per candidate cell line (thinly-profiled lines are visible):")
for line in CELL_LINES_PROBE:
    print("  %-7s %7d signatures" % (line, int((trt["_CELL"] == line).sum())))
print("LINCS InChIKeys with a pert mapping:", len(set(pert_to_ik.values())))


In [ ]:
# 6) OVERLAP per context + SUMMARY + recommendation -> single COPY-BACK block.
AR_IK = set(AR_LABELS)

def context_overlap(lines):
    sub = trt[trt["_CELL"].isin([x.upper() for x in lines])]
    pids = set(sub[SIG_PERT].astype(str))
    ctx_iks = {pert_to_ik[p] for p in pids if p in pert_to_ik}
    ov = ctx_iks & AR_IK
    pos = sum(AR_LABELS[k] == 1 for k in ov)
    neg = sum(AR_LABELS[k] == 0 for k in ov)
    return len(ov), pos, neg

def verdict(ov, pos, neg):
    ok = ov >= MIN_OVERLAP and min(pos, neg) >= MIN_COMPOUNDS_PER_CLASS
    why = []
    if ov < MIN_OVERLAP: why.append("overlap %d<%d" % (ov, MIN_OVERLAP))
    if min(pos, neg) < MIN_COMPOUNDS_PER_CLASS:
        why.append("min-class %d<%d" % (min(pos, neg), MIN_COMPOUNDS_PER_CLASS))
    return ("PASS" if ok else "failed_qc"), ("; ".join(why) if why else "clears both floors")

rows = []
for name, lines in CONTEXTS.items():
    ov, pos, neg = context_overlap(lines)
    prev = (pos / ov) if ov else 0.0
    vrd, why = verdict(ov, pos, neg)
    rows.append((name, ov, pos, neg, prev, vrd, why))

andro = [r for r in rows if "androgen" in r[0] and r[5] == "PASS"]
passing = [r for r in rows if r[5] == "PASS"]
if andro:
    rec = max(andro, key=lambda r: min(r[2], r[3]))
    rec_note = "androgen-relevant context clears the gate — biologically apt."
elif passing:
    rec = max(passing, key=lambda r: min(r[2], r[3]))
    rec_note = ("no androgen-relevant context clears the gate; using a broader/ER-style "
                "context — NOTE the biological-context limitation (not AR-tissue specific).")
else:
    rec = max(rows, key=lambda r: (r[1], min(r[2], r[3])))
    rec_note = ("NO context clears the gate at current floors -> expected outcome is "
                "failed_qc (a VALID M6 result: the gate rejects underpowered AR data).")

line = "=" * 72
print("\n" + line)
print("COPY-BACK SUMMARY  (AR Stop-2 coverage diagnostic — paste this whole block back)")
print(line)
print("SOURCE (measured, experimental — NOT consensus predictions):")
print("  source/table : %s | %s" % (SRC_TAG, SRC_KEY))
print("  structure col: %s   AR activity call column: %s" % (INCHI_COL or SMILES_COL, CALL_COL))
print("  (consensus-prediction article %s NOT used; predicted columns filtered)"
      % COMPARA_CONSENSUS_PREDICTION_ARTICLE)
print("LABELS:")
print("  clean AR structures: %d  (pos=%d, neg=%d)  | conflicts excluded: %d"
      % (len(AR_LABELS), n_pos, n_neg, conflicts))
print("GATE FLOORS: min_overlap=%d, min_compounds_per_class=%d" % (MIN_OVERLAP, MIN_COMPOUNDS_PER_CLASS))
print("trt_cp signatures/line: " + ", ".join(
    "%s=%d" % (l, int((trt['_CELL'] == l).sum())) for l in CELL_LINES_PROBE))
print(line)
print("%-28s %7s %6s %6s %7s  %-9s %s" % ("CONTEXT", "overlap", "pos", "neg", "prev", "VERDICT", "why"))
print("-" * 72)
for name, ov, pos, neg, prev, vrd, why in rows:
    print("%-28s %7d %6d %6d %6.1f%%  %-9s %s" % (name, ov, pos, neg, 100 * prev, vrd, why))
print(line)
print("RECOMMENDED CONTEXT: %s  [%s]" % (rec[0], rec[5]))
print("  rationale: %s" % rec_note)
print("  overlap=%d pos=%d neg=%d prevalence=%.1f%%" % (rec[1], rec[2], rec[3], 100 * rec[4]))
print(line)
print("END COPY-BACK SUMMARY")
print(line)
